# 💰 Financial Analysis — V3 Direct Net Flow Model
**Bike Share Toronto | GOLD_V2 Spatiotemporal Dataset**

Evaluates the V3 direct net_flow model and translates prediction accuracy into
daily system-wide rebalancing costs, operational savings, and P&L impact.

**Data source:** 2024 Bike Share Toronto Business Review (Toronto Parking Authority, Nov 29 2024)

**Note on revenue approach:** V3 predicts net_flow (arrivals − departures), not departures directly.
Revenue is computed from actual departures in the feature store.
The model's financial value is expressed primarily through rebalancing cost savings
and operational accuracy — a stronger and more defensible business case for a bike share system.

| Cell | Purpose |
|------|---------|
| 1 | Config — financial constants, paths, date range |
| 2 | Load V3 features + score model (actual net_flow + predictions) |
| 3 | Revenue & cost engine (actual trips + net_flow error impact) |
| 4 | Daily system-wide P&L table |
| 5 | Rebalancing cost layer (core value of V3) |
| 6 | Summary stats + dollar value of V3 model accuracy |
| 7 | Visualisations |

In [0]:
%pip install xgboost==2.0.3
%pip install lightgbm==4.3.0
%restart_python

## Cell 1 — Configuration

Sets all financial constants, file paths, and analysis parameters used throughout the notebook.

### Data Source
All financial constants are sourced from the **Bike Share Toronto 2024 Business Review** (Toronto Parking Authority Board, November 29, 2024):  
https://www.toronto.ca/legdocs/mmis/2024/pa/bgrd/backgroundfile-251202.pdf

### Analysis Window
The analysis covers **October 1, 2023 → September 30, 2024** — a full 12-month rolling year including the 2024 leap day.

### Revenue Constants *(Business Review, p. 7)*
| Parameter | Value | Description |
|---|---|---|
| Casual trip revenue | \$5.19 / trip | Pay-per-use riders |
| Annual member revenue | \$0.99 / trip | Subscription members |
| Casual mix — weekday | 18% | Member-dominated days |
| Casual mix — weekend | 38% | Leisure-dominated days |
| System blended casual share | 24% | Full-year average |

### Cost Structure *(Business Review, pp. 6–7)*
- **Variable cost:** \$1.98/trip (labour, maintenance, ops)
- **Total annual expenses:** \$16,068,000 CAD
- **Fixed overhead:** derived as `Total Expenses − (Total Rides × Variable Cost)` ≈ \$2.41M/year → ~\$6,591/day

### Rebalancing Cost Derivation
The rebalancing budget is anchored to **Shaheen et al. (2012)**, Mineta Transportation Institute, which reported an average cost of US\$667/station/month across early US bike share systems.  
Formula applied: `US$667 × 1.36 (USD→CAD) × 1,008 stations × 12 months ≈ CA$11.0M/year`

> **Note on inflation:** Inflation adjustment is intentionally omitted. The 2012 systems surveyed were small (avg ~10 stations) and manually operated. BST at 1,008 stations benefits from economies of scale, route optimisation software, and dedicated fleets — factors that offset or exceed 14 years of raw inflation.

### Net Flow Rebalancing Thresholds *(BST standard: 15-dock stations)*
| Threshold | Condition | Rationale |
|---|---|---|
| Over-stock | net flow > +13 | ≥87% full, ≤1 dock remaining — overflow imminent |
| Stock-out | net flow < −5 | Station losing 5+ bikes/hour — empty risk |

### Downtown Bounding Box
Analysis is scoped to downtown Toronto stations: lat `43.63–43.67`, lon `−79.41–−79.37`.

### DBFS Model Artifacts
The cell resolves all required paths for the V3 Direct Net Flow model:
- **Feature store** (`goldv2_features_netflow_v3_cyclic_stationtrend`)
- **Best model metadata** (selects XGB or LGBM winner from training)
- **XGBoost model** (`.json`) and **LightGBM model** (`.txt`)
- **Financial output** directory for saving daily P&L results


In [0]:

ANALYSIS_START = "2023-10-01"
ANALYSIS_END   = "2024-09-30"

REVENUE_PER_TRIP_CASUAL = 5.19   # $/trip — casual users
REVENUE_PER_TRIP_ANNUAL = 0.99   # $/trip — annual members
CASUAL_MIX_WEEKDAY      = 0.18   # casual share weekdays (member-dominated)
CASUAL_MIX_WEEKEND      = 0.38   # casual share weekends (leisure-dominated)
# System blended average = 24% casual (page 7)

# Total expenses: $16,068K | Total rides: 6.9M
# Cost/trip $1.98 = variable cost per trip (labour, maintenance, ops)
# Fixed overhead = $16.068M - (6.9M x $1.98) = ~$2.41M/year
#   → $2,406,000 / 365 = ~$6,591/day
COST_PER_TRIP_VARIABLE  = 1.98          # $/trip — variable cost 
TOTAL_ANNUAL_EXPENSES   = 16_068_000    # $ CAD — full year 2024 forecast 
TOTAL_ANNUAL_RIDES      = 6_900_000     # rides — full year 2024 forecast
FIXED_COST_ANNUAL       = round(TOTAL_ANNUAL_EXPENSES - (TOTAL_ANNUAL_RIDES * COST_PER_TRIP_VARIABLE), 2)
FIXED_COST_DAILY        = round(FIXED_COST_ANNUAL / 365, 2)

# Source: Shaheen et al. (2012), Mineta Transportation Institute
# Formula: US$667 × 1.36 USD/CAD × 1,008 stations × 12 months
USD_TO_CAD         = 1.36
REBAL_BASE_USD     = 667.0
N_STATIONS         = 1008
REBAL_ANNUAL_CAD   = REBAL_BASE_USD * USD_TO_CAD * N_STATIONS * 12
# = 667 × 1.36 × 1008 × 12 = ~$11.0M CAD/year

# REBAL_COST_PER_EVENT is derived in Cell 5 once event counts are known.
REBAL_COST_PER_EVENT = None

# BST standard station capacity: 15 docks
OVER_STOCK_THRESHOLD = 13    # predicted net > +13 → station ≥87% full → overflow risk
STOCK_OUT_THRESHOLD  = -5    # predicted net < -5  → station draining fast → empty risk

DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX = 43.63, 43.67
DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX = -79.41, -79.37

FEATURES_DIR        = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/experiments/features/"
    "goldv2_features_netflow_v3_cyclic_stationtrend"
)
BEST_MODEL_META_JSON = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/metadata/"
    "netflow_direct_v3_cyclic_stationtrend/best_model_netflow_direct_v3_cyclic_stationtrend.json"
)
FEATURE_META_JSON   = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/metadata/"
    "netflow_direct_v3_cyclic_stationtrend/features_netflow_direct_v3_cyclic_stationtrend.json"
)
XGB_MODEL_JSON      = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/models/"
    "netflow_direct_v3_cyclic_stationtrend/netflow_direct_xgb_weight10_v3_cyclic_stationtrend.json"
)
LGBM_MODEL_TXT      = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/models/"
    "netflow_direct_v3_cyclic_stationtrend/netflow_direct_lgbm_weight10_v3_cyclic_stationtrend.txt"
)
FINANCIAL_OUT       = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/"
    "experiments/financial/goldv2_financial_analysis_v3_netflow"
)

blended_rev_weekday = (CASUAL_MIX_WEEKDAY * REVENUE_PER_TRIP_CASUAL +
                       (1 - CASUAL_MIX_WEEKDAY) * REVENUE_PER_TRIP_ANNUAL)
blended_rev_weekend = (CASUAL_MIX_WEEKEND * REVENUE_PER_TRIP_CASUAL +
                       (1 - CASUAL_MIX_WEEKEND) * REVENUE_PER_TRIP_ANNUAL)
blended_rev_system  = (0.24 * REVENUE_PER_TRIP_CASUAL +
                       0.76 * REVENUE_PER_TRIP_ANNUAL)

print("=" * 60)
print(" FINANCIAL ANALYSIS CONFIG — V3 DIRECT NET FLOW")
print("=" * 60)
print(f" Period                   : {ANALYSIS_START} → {ANALYSIS_END}")
print(f" Revenue/trip casual      : ${REVENUE_PER_TRIP_CASUAL:.2f}")
print(f" Revenue/trip annual      : ${REVENUE_PER_TRIP_ANNUAL:.2f}")
print(f" Blended rev weekday      : ${blended_rev_weekday:.4f}/trip")
print(f" Blended rev weekend      : ${blended_rev_weekend:.4f}/trip")
print(f" Blended rev system       : ${blended_rev_system:.4f}/trip")
print(f"")
print(f" Variable cost/trip       : ${COST_PER_TRIP_VARIABLE:.2f}  (Business Review p.7)")
print(f" Total annual expenses    : ${TOTAL_ANNUAL_EXPENSES:,.0f}")
print(f" Fixed overhead/year      : ${FIXED_COST_ANNUAL:,.2f}  (derived)")
print(f" Fixed overhead/day       : ${FIXED_COST_DAILY:,.2f}  (used in daily P&L)")
print(f"")
print(f" Rebal budget/year (CAD)  : ${REBAL_ANNUAL_CAD:,.2f}")
print(f" Rebal % of total costs   : {REBAL_ANNUAL_CAD/TOTAL_ANNUAL_EXPENSES*100:.1f}%")
print(f" Rebal cost/event         : derived in Cell 5")
print(f" Station capacity         : 15 docks (BST operational standard)")
print(f" Over-stock threshold     : net flow > +{OVER_STOCK_THRESHOLD} bikes (≥87% full, ≤1 dock left)")
print(f" Stock-out threshold      : net flow < {STOCK_OUT_THRESHOLD} bikes (draining fast, empty risk)")
print(f"")
print(f" Model                    : V3 Direct Net Flow (XGB or LGBM — best wins)")
print(f" Features dir             : {FEATURES_DIR}")
print("=" * 60)

## Cell 2 — Load V3 Features & Score Both Models

Loads the V3 feature store from DBFS, scores both XGBoost and LightGBM models, and joins actual departure and arrival counts from the Gold V2 dataset.

### Why Join Gold V2?
V3 was trained to predict `net_flow` (arrivals − departures) directly. As a result, the V3 feature store **does not contain raw departure or arrival columns** — they were not needed for training. Gold V2 is joined back on `(station_id, year, month, day, hour)` to recover actual departures and arrivals, which form the revenue base in Cell 3.

### Scoring Both Models
Both XGBoost and LightGBM are scored in parallel for head-to-head comparison across the full analysis window. The best model was selected during training and recorded in metadata; however, both predictions are retained here so every downstream cell can report savings and error metrics for each.

### Output: `scored_pdf`
One row per station per hour, containing:

| Column | Description |
|---|---|
| `net_flow` | Ground-truth net flow (arrivals − departures) |
| `xgb_net_pred` | XGBoost V3 prediction |
| `lgbm_net_pred` | LightGBM V3 prediction |
| `departures` | Actual departures (joined from Gold V2) |
| `arrivals` | Actual arrivals (joined from Gold V2) |

### Key Steps
1. **Validate all artifact paths** — raises an exception early if any DBFS path is missing.
2. **Load model metadata** — reads best model selection and feature list from JSON.
3. **Load both models** — XGBoost (`.json`) and LightGBM (`.txt`) are loaded into memory.
4. **Filter by analysis window** — month-level partition filter applied before loading Parquet.
5. **Derive helper columns** — `station_bucket` (hash-based), `is_weekend`, timestamp alignment.
6. **Join Gold V2** — left join on station + time keys; unmatched rows default departures to 0 with a warning.
7. **Score XGBoost** — feature matrix passed through `xgb.DMatrix`.
8. **Score LightGBM** — same feature matrix passed directly to `lgb.Booster.predict()`.


In [0]:

from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import json, zlib
import xgboost as xgb
import lightgbm as lgb

GOLD_V2_DIR = (
    "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/"
    "data/gold/gold_v2_spatiotemporal_events"
)

def path_exists(p):
    try: dbutils.fs.ls(p); return True
    except: return False

def read_dbfs_text(p, mb=60_000_000):
    return dbutils.fs.head(p, mb)

def station_to_bucket(sid, n=512):
    return zlib.crc32(str(sid).encode()) % n if sid else 0

def ensure_features(pdf, feats, fill=0.0):
    for c in feats:
        if c not in pdf.columns:
            pdf[c] = fill
        pdf[c] = pd.to_numeric(pdf[c], errors="coerce").fillna(fill)
    return pdf

required = [
    FEATURES_DIR, BEST_MODEL_META_JSON, FEATURE_META_JSON,
    XGB_MODEL_JSON, LGBM_MODEL_TXT, GOLD_V2_DIR
]
for p in required:
    if not path_exists(p):
        raise Exception(f"Missing artifact: {p}")
print("All artifact paths exist")

best_meta    = json.loads(read_dbfs_text(BEST_MODEL_META_JSON, 500_000))
feat_meta    = json.loads(read_dbfs_text(FEATURE_META_JSON,    500_000))

BEST_MODEL   = best_meta["best_model"]
FEATURES     = feat_meta["netflow_features"]
HASH_BUCKETS = int(feat_meta.get("hash_buckets", 512))

print(f"Best model selected during training : {BEST_MODEL}")
print(f"Feature count                       : {len(FEATURES)}")

xgb_booster = xgb.Booster()
xgb_booster.load_model(
    bytearray(read_dbfs_text(XGB_MODEL_JSON).encode("latin-1"))
)
lgbm_booster = lgb.Booster(model_str=read_dbfs_text(LGBM_MODEL_TXT))
print("Both models loaded (XGB + LGBM)")

start_ts = pd.Timestamp(ANALYSIS_START)
end_ts   = pd.Timestamp(ANALYSIS_END) + pd.Timedelta(hours=23)

sim_months = pd.period_range(
    start=start_ts.to_period("M"),
    end=end_ts.to_period("M"),
    freq="M"
)
month_filter = None
for p in sim_months:
    cond = (F.col("year") == int(p.year)) & (F.col("month") == int(p.month))
    month_filter = cond if month_filter is None else month_filter | cond

id_cols  = ["station_id", "year", "month", "day", "hour", "date", "dow_num"]
all_want = list(dict.fromkeys(id_cols + ["net_flow"] + FEATURES))

df_feat = spark.read.parquet(FEATURES_DIR).filter(month_filter)
avail   = set(df_feat.columns)
select  = [c for c in all_want if c in avail]

raw_pdf = df_feat.select(select).toPandas()
raw_pdf["date"]       = pd.to_datetime(raw_pdf["date"])
raw_pdf["station_id"] = raw_pdf["station_id"].astype(str)
raw_pdf["ts_hour"]    = raw_pdf["date"] + pd.to_timedelta(raw_pdf["hour"], unit="h")

# Filter to exact analysis window
raw_pdf = raw_pdf[
    (raw_pdf["ts_hour"] >= start_ts) &
    (raw_pdf["ts_hour"] <= end_ts)
].copy().reset_index(drop=True)

# Derived columns
raw_pdf["station_bucket"] = raw_pdf["station_id"].map(
    lambda s: station_to_bucket(s)
).astype(np.int16)
raw_pdf["is_weekend"] = raw_pdf["dow_num"].isin([1, 7]).astype(np.int8)
raw_pdf = ensure_features(raw_pdf, FEATURES)

print(f"V3 feature store: {len(raw_pdf):,} station-hours, "
      f"{raw_pdf['station_id'].nunique()} stations")

# Gold V2 has the original departures and arrivals columns
# that the V3 feature store dropped since it only needed net_flow.
df_gold = (
    spark.read.parquet(GOLD_V2_DIR)
    .filter(month_filter)
    .filter(
        (F.col("lat").between(DOWNTOWN_LAT_MIN, DOWNTOWN_LAT_MAX)) &
        (F.col("lon").between(DOWNTOWN_LON_MIN, DOWNTOWN_LON_MAX))
    )
    .select("station_id", "year", "month", "day", "hour",
            "departures", "arrivals")
)

gold_pdf = df_gold.toPandas()
gold_pdf["station_id"] = gold_pdf["station_id"].astype(str)
gold_pdf["departures"]  = pd.to_numeric(gold_pdf["departures"],  errors="coerce").fillna(0)
gold_pdf["arrivals"]    = pd.to_numeric(gold_pdf["arrivals"],    errors="coerce").fillna(0)

print(f"Gold V2 rows loaded: {len(gold_pdf):,}")

# Merge on station + time keys
join_keys = ["station_id", "year", "month", "day", "hour"]
raw_pdf = raw_pdf.merge(gold_pdf[join_keys + ["departures", "arrivals"]],
                        on=join_keys, how="left")

# Fill any unmatched rows with 0 (shouldn't happen if date ranges align)
raw_pdf["departures"] = raw_pdf["departures"].fillna(0)
raw_pdf["arrivals"]   = raw_pdf["arrivals"].fillna(0)

unmatched = (raw_pdf["departures"] == 0).sum()
if unmatched > 0:
    print(f"WARNING: {unmatched:,} rows had no Gold V2 match — departures set to 0")
else:
    print("All rows matched successfully to Gold V2 departures/arrivals")

X    = raw_pdf[FEATURES].astype(np.float32).values
dmat = xgb.DMatrix(X, feature_names=FEATURES)
raw_pdf["xgb_net_pred"] = xgb_booster.predict(dmat).astype(np.float32)
print("XGBoost scored")

raw_pdf["lgbm_net_pred"] = lgbm_booster.predict(X).astype(np.float32)
print("LightGBM scored")

scored_pdf = raw_pdf.copy()
print(f"\nScoring complete.")
print(f"  Shape             : {scored_pdf.shape}")
print(f"  Departures range  : {scored_pdf['departures'].min():.0f} – {scored_pdf['departures'].max():.0f}")
print(f"  Net flow range    : {scored_pdf['net_flow'].min():.0f} – {scored_pdf['net_flow'].max():.0f}")
print(f"  XGB pred range    : {scored_pdf['xgb_net_pred'].min():.2f} – {scored_pdf['xgb_net_pred'].max():.2f}")
print(f"  LGBM pred range   : {scored_pdf['lgbm_net_pred'].min():.2f} – {scored_pdf['lgbm_net_pred'].max():.2f}")

## Cell 3 — Revenue & Cost Engine

Applies the financial model to each station-hour row, computing actual revenue, variable cost, gross profit, and a model-specific **revenue-at-risk** metric.

### Revenue Calculation
Revenue is always computed from **actual departures** — not from model predictions. V3 predicts `net_flow`, not departures, so the model's direct financial value is expressed through **operational accuracy** (rebalancing), not revenue generation.

The blended revenue rate varies by day type:
- **Weekday:** `18% × $5.19 + 82% × $0.99`
- **Weekend:** `38% × $5.19 + 62% × $0.99`

### Cost Breakdown
| Component | Value | Source |
|---|---|---|
| Variable cost/trip | \$1.98 | Business Review p. 7 |
| Fixed cost/day | ~\$6,591 | Derived in Cell 1 |
| Rebalancing cost | Calculated in Cell 5 | Separate operational layer |

### Revenue at Risk from Net Flow Error
A `revenue_at_risk` column is computed per model as:
```
revenue_at_risk = |predicted_net_flow − actual_net_flow| × blended_revenue_rate
```
This estimates the revenue exposed when a model error causes a missed rebalancing event — e.g., a station unexpectedly runs empty (stock-out) because the model underestimated outflow. A lower value indicates a model that keeps more trips available.

> **Note:** Revenue at risk is a conservative proxy, not a direct lost-revenue figure. It measures prediction error in revenue terms, complementing the dispatch-cost metric in Cell 5.

### Outputs Added to `fin`
| Column | Description |
|---|---|
| `actual_revenue` | Departures × blended revenue rate |
| `actual_var_cost` | Departures × \$1.98 |
| `actual_profit` | Revenue − variable cost |
| `xgb_net_flow_error` | Absolute XGB prediction error |
| `lgbm_net_flow_error` | Absolute LGBM prediction error |
| `xgb_revenue_at_risk` | XGB error × revenue rate |
| `lgbm_revenue_at_risk` | LGBM error × revenue rate |
| `revenue_at_risk_delta` | XGB risk − LGBM risk (positive = LGBM is safer) |


In [0]:

def revenue_per_trip(is_weekend_series):
    return np.where(
        is_weekend_series == 1,
        CASUAL_MIX_WEEKEND * REVENUE_PER_TRIP_CASUAL +
            (1 - CASUAL_MIX_WEEKEND) * REVENUE_PER_TRIP_ANNUAL,
        CASUAL_MIX_WEEKDAY * REVENUE_PER_TRIP_CASUAL +
            (1 - CASUAL_MIX_WEEKDAY) * REVENUE_PER_TRIP_ANNUAL
    )

fin = scored_pdf.copy()
fin["rev_rate"] = revenue_per_trip(fin["is_weekend"])

fin["actual_revenue"]  = fin["departures"]  * fin["rev_rate"]
fin["actual_var_cost"] = fin["departures"]  * COST_PER_TRIP_VARIABLE
fin["actual_profit"]   = fin["actual_revenue"] - fin["actual_var_cost"]

# This measures how far off each model was in predicting net_flow.
# A large error means the model may have triggered a wrong
# rebalancing decision, which is costed in Cell 5.
fin["xgb_net_flow_error"]  = (fin["xgb_net_pred"]  - fin["net_flow"]).abs()
fin["lgbm_net_flow_error"] = (fin["lgbm_net_pred"] - fin["net_flow"]).abs()

# When a station unexpectedly runs empty (stock-out missed by model),
# the lost trips can be estimated as the absolute net flow error
# times the blended revenue rate.
fin["xgb_revenue_at_risk"]  = fin["xgb_net_flow_error"]  * fin["rev_rate"]
fin["lgbm_revenue_at_risk"] = fin["lgbm_net_flow_error"] * fin["rev_rate"]
fin["revenue_at_risk_delta"] = fin["xgb_revenue_at_risk"] - fin["lgbm_revenue_at_risk"]

print("Revenue & cost engine applied.")
print(f"  Rev rate weekday           : ${blended_rev_weekday:.4f}/trip")
print(f"  Rev rate weekend           : ${blended_rev_weekend:.4f}/trip")
print(f"  Variable cost              : ${COST_PER_TRIP_VARIABLE:.2f}/trip")
print(f"  Fixed cost/day             : ${FIXED_COST_DAILY:,.2f}")
print(f"  Rows processed             : {len(fin):,}")
print()
analysis_days = (pd.Timestamp(ANALYSIS_END) - pd.Timestamp(ANALYSIS_START)).days + 1
total_fixed   = FIXED_COST_DAILY * analysis_days
print("Sanity check — system totals across full window:")
print(f"  Actual total revenue       : ${fin['actual_revenue'].sum():>12,.2f}")
print(f"  Actual total var cost      : ${fin['actual_var_cost'].sum():>12,.2f}")
print(f"  Fixed cost ({analysis_days} days)       : ${total_fixed:>12,.2f}")
print(f"  Actual gross profit        : ${fin['actual_profit'].sum():>12,.2f}")
print(f"  Actual net profit          : ${fin['actual_profit'].sum() - total_fixed:>12,.2f}")
print(f"  XGB  total revenue at risk : ${fin['xgb_revenue_at_risk'].sum():>12,.2f}")
print(f"  LGBM total revenue at risk : ${fin['lgbm_revenue_at_risk'].sum():>12,.2f}")

## Cell 4 — Daily System-Wide P&L Table

Aggregates all downtown station-hours into a daily P&L view, applies the fixed daily cost, and flags which model had lower net flow MAE each day.

### Aggregation Logic
All rows in `fin` are grouped by `date`. Each day produces:
- **Actual trips, revenue, variable cost, gross profit** — summed across all downtown stations
- **Fixed cost** — applied as a flat \$6,591/day
- **Net profit** — gross profit minus fixed cost
- **Net flow MAE** — mean absolute error per station-hour for XGB and LGBM
- **Revenue at risk** — summed across stations for each model
- **Better model flag** — `'LGBM'` or `'XGB'` based on which had lower MAE that day

> **Note:** Rebalancing cost is a **separate operational layer** added in Cell 5. It is not included here to avoid double-counting with the \$1.98/trip variable cost, which already embeds base rebalancing expenses.

### Monthly Rollup
A `year_month` column is derived and used to produce a monthly summary showing:
- Monthly trips, revenue, and net profit
- Revenue at risk per model
- Average net flow MAE per model
- Days where LGBM outperformed XGB (count and percentage)


In [0]:

analysis_days = (pd.Timestamp(ANALYSIS_END) - pd.Timestamp(ANALYSIS_START)).days + 1

daily = (
    fin
    .groupby("date", as_index=False)
    .agg(
        actual_trips           = ("departures",            "sum"),
        actual_revenue         = ("actual_revenue",        "sum"),
        actual_var_cost        = ("actual_var_cost",       "sum"),
        actual_profit          = ("actual_profit",         "sum"),
        actual_net_flow        = ("net_flow",              "sum"),
        xgb_net_flow_sum       = ("xgb_net_pred",          "sum"),
        lgbm_net_flow_sum      = ("lgbm_net_pred",         "sum"),
        xgb_net_flow_mae       = ("xgb_net_flow_error",    "mean"),
        lgbm_net_flow_mae      = ("lgbm_net_flow_error",   "mean"),
        xgb_revenue_at_risk    = ("xgb_revenue_at_risk",   "sum"),
        lgbm_revenue_at_risk   = ("lgbm_revenue_at_risk",  "sum"),
        revenue_at_risk_delta  = ("revenue_at_risk_delta", "sum"),
        stations               = ("station_id",            "nunique"),
        is_weekend             = ("is_weekend",            "max"),
    )
    .sort_values("date")
    .reset_index(drop=True)
)

# Apply fixed daily cost
daily["actual_fixed_cost"] = FIXED_COST_DAILY
daily["actual_net_profit"] = daily["actual_profit"] - FIXED_COST_DAILY

# Flag which model had lower net flow MAE each day
daily["better_model"] = np.where(
    daily["lgbm_net_flow_mae"] < daily["xgb_net_flow_mae"], "LGBM", "XGB"
)

print("Daily System-Wide P&L ($ CAD | V3 Net Flow Model)")
print(f"{'Date':<12} {'ActTrips':>9} {'ActRev':>9} {'VarCost':>8} {'FixCost':>8} "
      f"{'NetPft':>9} {'XGBRisk':>9} {'LGBRisk':>9} {'RiskSave':>9} {'Better':>6}")
print("-" * 105)
for _, r in daily.iterrows():
    print(
        f"{str(r['date'])[:10]:<12} "
        f"{r['actual_trips']:>9,.0f} "
        f"${r['actual_revenue']:>8,.2f} "
        f"${r['actual_var_cost']:>7,.2f} "
        f"${r['actual_fixed_cost']:>7,.2f} "
        f"${r['actual_net_profit']:>8,.2f} "
        f"${r['xgb_revenue_at_risk']:>8,.2f} "
        f"${r['lgbm_revenue_at_risk']:>8,.2f} "
        f"${r['revenue_at_risk_delta']:>8,.2f} "
        f"{r['better_model']:>6}"
    )

daily["year_month"] = pd.to_datetime(daily["date"]).dt.to_period("M").astype(str)
monthly = (
    daily
    .groupby("year_month", as_index=False)
    .agg(
        actual_trips           = ("actual_trips",          "sum"),
        actual_revenue         = ("actual_revenue",        "sum"),
        actual_net_profit      = ("actual_net_profit",     "sum"),
        xgb_revenue_at_risk    = ("xgb_revenue_at_risk",   "sum"),
        lgbm_revenue_at_risk   = ("lgbm_revenue_at_risk",  "sum"),
        xgb_net_flow_mae       = ("xgb_net_flow_mae",      "mean"),
        lgbm_net_flow_mae      = ("lgbm_net_flow_mae",     "mean"),
        lgbm_wins_days         = ("better_model",          lambda x: (x == "LGBM").sum()),
        total_days             = ("date",                  "count"),
    )
)
monthly["lgbm_win_pct"] = (monthly["lgbm_wins_days"] / monthly["total_days"] * 100).round(1)
print("\nMonthly P&L Rollup:")
display(monthly)

## Cell 5 — Rebalancing Cost Layer

This is the **core financial value proposition of V3**. It quantifies the operational cost of incorrect model predictions by counting unnecessary truck dispatches triggered when a model predicted a threshold breach that did not actually occur.

### Why a Standalone Metric?
Rebalancing cost is presented separately from the P&L in Cell 4 to avoid double-counting: the \$1.98/trip variable cost already includes base rebalancing expenditure. This cell measures only the **incremental cost** from *wrong dispatch decisions* caused by model errors.

### Cost-per-Event Derivation
Rather than assuming an external dispatch frequency, the cost per event is derived entirely from BST ground-truth data:

1. **Annual rebalancing budget** — `US$667 × 1.36 × 1,008 × 12 ≈ CA$10,972,523/year` (Shaheen et al. 2012)
2. **Annual dispatch event count** — all station-hours where actual `net_flow` exceeded a threshold are counted and annualised to 365 days
3. **Cost per event** — `annual budget ÷ annual events`

This approach is **fully traceable** with no invented assumptions:
- Budget source: Shaheen et al. (2012), Mineta Transportation Institute
- Event count: derived from actual BST Gold V2 net flow data

### Thresholds *(BST 15-dock capacity standard)*
| Threshold | Condition | Meaning |
|---|---|---|
| Over-stock | net flow > +13 | Station ≥87% full (≤1 dock remaining) |
| Stock-out | net flow < −5 | Station losing 5+ bikes/hour |

### Model Error Classification
A **prediction error** is flagged when the model predicts a threshold breach that does **not** occur in reality — i.e., a false alarm causing an unnecessary dispatch:
- `overstock_error`: model predicted over-stock, but actual net flow was within normal range
- `stockout_error`: model predicted stock-out, but actual net flow was within normal range

Each error event is costed at `REBAL_COST_PER_EVENT` (derived above).

### Steps
1. Derive `REBAL_COST_PER_EVENT` from actual BST data
2. Flag over-stock and stock-out prediction errors for XGB and LGBM
3. Apply cost per event to each error
4. Aggregate daily rebalancing costs per model
5. Compute `rebal_cost_saved` = XGB cost − LGBM cost (positive = LGBM saves money)


In [0]:

rebal = fin.copy()

rebal["actual_net_flow"] = rebal["net_flow"]

# Count all station-hours where a real rebalancing trigger occurred
actual_overstock_events = int((rebal["actual_net_flow"] >  OVER_STOCK_THRESHOLD).sum())
actual_stockout_events  = int((rebal["actual_net_flow"] <  STOCK_OUT_THRESHOLD).sum())
actual_total_events     = actual_overstock_events + actual_stockout_events

# Annualise: data covers 366 days (Oct 2023 – Sep 2024 incl. leap day)
analysis_days_rebal  = (pd.Timestamp(ANALYSIS_END) - pd.Timestamp(ANALYSIS_START)).days + 1
annual_events        = round(actual_total_events * (365 / analysis_days_rebal))

# Cost per event = annual budget ÷ annual event count
REBAL_COST_PER_EVENT = round(REBAL_ANNUAL_CAD / annual_events, 2)

print("=" * 65)
print(" REBALANCING EVENT DERIVATION (from actual BST data)")
print("=" * 65)
print(f" Station capacity            : 15 docks (BST standard)")
print(f" Over-stock threshold        : net flow > +{OVER_STOCK_THRESHOLD} (≥87% full, ≤1 dock left)")
print(f" Stock-out threshold         : net flow < {STOCK_OUT_THRESHOLD}  (losing 5+ bikes/hr)")
print()
print(f" Actual over-stock events    : {actual_overstock_events:>10,}")
print(f" Actual stock-out events     : {actual_stockout_events:>10,}")
print(f" Total actual events         : {actual_total_events:>10,}  (over {analysis_days_rebal} days)")
print(f" Annualised events (365d)    : {annual_events:>10,}")
print()
print(f" Rebal annual budget (CAD)   : ${REBAL_ANNUAL_CAD:>12,.2f}  (Shaheen et al. 2012)")
print(f" Derived cost per event      : ${REBAL_COST_PER_EVENT:>12,.2f}  (budget ÷ annual events)")
print("=" * 65)
print()

# Error = model predicted a threshold breach that did NOT happen
# → unnecessary dispatch triggered by wrong prediction
rebal["xgb_overstock_error"]  = (
    (rebal["xgb_net_pred"]   >  OVER_STOCK_THRESHOLD) &
    (rebal["actual_net_flow"] <= OVER_STOCK_THRESHOLD)
).astype(int)
rebal["lgbm_overstock_error"] = (
    (rebal["lgbm_net_pred"]  >  OVER_STOCK_THRESHOLD) &
    (rebal["actual_net_flow"] <= OVER_STOCK_THRESHOLD)
).astype(int)
rebal["xgb_stockout_error"]   = (
    (rebal["xgb_net_pred"]   <  STOCK_OUT_THRESHOLD) &
    (rebal["actual_net_flow"] >= STOCK_OUT_THRESHOLD)
).astype(int)
rebal["lgbm_stockout_error"]  = (
    (rebal["lgbm_net_pred"]  <  STOCK_OUT_THRESHOLD) &
    (rebal["actual_net_flow"] >= STOCK_OUT_THRESHOLD)
).astype(int)
rebal["xgb_rebal_error"]  = (
    (rebal["xgb_overstock_error"]  == 1) | (rebal["xgb_stockout_error"]  == 1)
).astype(int)
rebal["lgbm_rebal_error"] = (
    (rebal["lgbm_overstock_error"] == 1) | (rebal["lgbm_stockout_error"] == 1)
).astype(int)

rebal["xgb_rebal_cost"]  = rebal["xgb_rebal_error"]  * REBAL_COST_PER_EVENT
rebal["lgbm_rebal_cost"] = rebal["lgbm_rebal_error"] * REBAL_COST_PER_EVENT

daily_rebal = (
    rebal.groupby("date", as_index=False)
    .agg(
        xgb_rebal_events      = ("xgb_rebal_error",      "sum"),
        xgb_overstock_events  = ("xgb_overstock_error",  "sum"),
        xgb_stockout_events   = ("xgb_stockout_error",   "sum"),
        xgb_rebal_cost        = ("xgb_rebal_cost",       "sum"),
        lgbm_rebal_events     = ("lgbm_rebal_error",     "sum"),
        lgbm_overstock_events = ("lgbm_overstock_error", "sum"),
        lgbm_stockout_events  = ("lgbm_stockout_error",  "sum"),
        lgbm_rebal_cost       = ("lgbm_rebal_cost",      "sum"),
    )
    .sort_values("date")
)
daily_rebal["rebal_cost_saved"] = (
    daily_rebal["xgb_rebal_cost"] - daily_rebal["lgbm_rebal_cost"]
)

total_xgb_rebal   = rebal["xgb_rebal_cost"].sum()
total_lgbm_rebal  = rebal["lgbm_rebal_cost"].sum()
total_xgb_events  = rebal["xgb_rebal_error"].sum()
total_lgbm_events = rebal["lgbm_rebal_error"].sum()

print("=" * 65)
print(" MODEL DISPATCH ERROR COMPARISON — XGB vs LGBM")
print("=" * 65)
print(f" Note: rebalancing cost is already embedded in the $1.98/trip")
print(f" variable cost. This measures INCREMENTAL cost from wrong")
print(f" dispatch decisions caused by model prediction errors.")
print(f" Cost per event: CA${REBAL_COST_PER_EVENT:,.2f} (Shaheen budget ÷ actual BST events)")
print()
print(f"                               {'XGBoost':>12}  {'LightGBM':>12}  {'Saving':>12}")
print(f" Wrong dispatch events  : {total_xgb_events:>12,}  {total_lgbm_events:>12,}  "
      f"{total_xgb_events - total_lgbm_events:>+12,}")
print(f"  → Over-stock errors   : {rebal['xgb_overstock_error'].sum():>12,}  "
      f"{rebal['lgbm_overstock_error'].sum():>12,}")
print(f"  → Stock-out errors    : {rebal['xgb_stockout_error'].sum():>12,}  "
      f"{rebal['lgbm_stockout_error'].sum():>12,}")
print(f" Total dispatch cost    : CA${total_xgb_rebal:>11,.2f}  "
      f"CA${total_lgbm_rebal:>11,.2f}  "
      f"CA${total_xgb_rebal - total_lgbm_rebal:>+11,.2f}")
print("=" * 65)

display(daily_rebal.head(30))

## Cell 6 — Summary Statistics & Dollar Value of V3 Model Accuracy

Combines the P&L results from Cell 4 and the rebalancing operational metric from Cell 5 into a single full-period summary. Quantifies the financial advantage of LGBM over XGB across three dimensions.

### Full-Period Summary Table
Covers the complete analysis window (`2023-10-01 → 2024-09-30`), reporting:
- **Actual system P&L:** total trips, revenue, variable cost, gross profit, fixed cost, net profit
- **Model prediction quality:** net flow MAE for XGB and LGBM
- **Revenue at risk:** total exposure from prediction errors, per model
- **Days where LGBM outperformed XGB** (count and percentage)
- **Rebalancing operational metric:** incorrect dispatch events and associated cost per model

### Dollar Value of V3 Model Accuracy (LGBM vs XGB)
The financial advantage of LGBM over XGB is expressed as three additive components:

| Component | Calculation |
|---|---|
| **Net flow MAE delta** | `XGB MAE − LGBM MAE` (bikes/station-hour) |
| **Revenue at risk improvement** | `MAE delta × avg revenue rate × total station-hours` |
| **Rebalancing dispatch saving** | `XGB dispatch cost − LGBM dispatch cost` |
| **Total LGBM advantage** | Sum of revenue at risk improvement + dispatch saving |

> The combined figure represents the estimated annual dollar value of choosing LGBM over XGB for this operational use case.

### Output
The daily P&L table is saved to DBFS at the path defined by `FINANCIAL_OUT` in Cell 1.


In [0]:

from sklearn.metrics import mean_absolute_error

actual_nf   = fin["net_flow"].values
xgb_mae     = mean_absolute_error(actual_nf, fin["xgb_net_pred"].values)
lgbm_mae    = mean_absolute_error(actual_nf, fin["lgbm_net_pred"].values)
mae_delta   = xgb_mae - lgbm_mae

analysis_days   = (pd.Timestamp(ANALYSIS_END) - pd.Timestamp(ANALYSIS_START)).days + 1
total_fixed     = FIXED_COST_DAILY * analysis_days
avg_rev_rate    = fin["rev_rate"].mean()
n_station_hours = len(fin)

actual_total_rev  = fin["actual_revenue"].sum()
actual_total_var  = fin["actual_var_cost"].sum()
actual_gross      = fin["actual_profit"].sum()
actual_net        = actual_gross - total_fixed

xgb_risk_total    = fin["xgb_revenue_at_risk"].sum()
lgbm_risk_total   = fin["lgbm_revenue_at_risk"].sum()
xgb_rebal_cost    = rebal["xgb_rebal_cost"].sum()
lgbm_rebal_cost   = rebal["lgbm_rebal_cost"].sum()

lgbm_wins         = (daily["better_model"] == "LGBM").sum()
total_days        = len(daily)

# Revenue at risk improvement from MAE delta on net flow
# Each unit of MAE improvement reduces operational risk
# proportionally to the revenue rate
rev_risk_improvement = mae_delta * avg_rev_rate * n_station_hours

print("=" * 70)
print(" FULL PERIOD FINANCIAL SUMMARY — V3 DIRECT NET FLOW MODEL")
print(f" {ANALYSIS_START} → {ANALYSIS_END}  ({analysis_days} days)")
print("=" * 70)
print(f"{'Metric':<40} {'Actual':>12}  {'XGBoost':>10}  {'LightGBM':>10}")
print("-" * 70)
print(f"{'Total trips (actual departures)':<40} {fin['departures'].sum():>12,.0f}  {'—':>10}  {'—':>10}")
print(f"{'Total revenue ($)':<40} ${actual_total_rev:>11,.0f}  {'—':>10}  {'—':>10}")
print(f"{'Variable cost @ $1.98/trip ($)':<40} ${actual_total_var:>11,.0f}  {'—':>10}  {'—':>10}")
print(f"{'Gross profit ($)':<40} ${actual_gross:>11,.0f}  {'—':>10}  {'—':>10}")
print(f"{'Fixed cost ($)':<40} ${total_fixed:>11,.0f}  {'(same)':>10}  {'(same)':>10}")
print(f"{'Net profit ($)':<40} ${actual_net:>11,.0f}  {'—':>10}  {'—':>10}")
print("-" * 70)
print(f"{'Net flow MAE (bikes/station-hour)':<40} {'—':>12}  {xgb_mae:>10.4f}  {lgbm_mae:>10.4f}")
print(f"{'Total revenue at risk ($)':<40} {'—':>12}  ${xgb_risk_total:>9,.0f}  ${lgbm_risk_total:>9,.0f}")
print(f"{'Days LGBM closer to actual':<40} {lgbm_wins}/{total_days} ({lgbm_wins/total_days*100:.1f}%)")
print()
print(" REBALANCING OPERATIONAL METRIC (standalone)")
print("-" * 70)
print(f"{'Incorrect dispatch events':<40} {'—':>12}  {rebal['xgb_rebal_error'].sum():>10,}  {rebal['lgbm_rebal_error'].sum():>10,}")
print(f"{'Estimated dispatch cost ($)':<40} {'—':>12}  ${xgb_rebal_cost:>9,.0f}  ${lgbm_rebal_cost:>9,.0f}")
print(f"{'Dispatch cost saving — LGBM ($)':<40} ${xgb_rebal_cost - lgbm_rebal_cost:>9,.0f}")
print("=" * 70)
print()
print(" DOLLAR VALUE OF V3 MODEL ACCURACY (LGBM vs XGB)")
print("-" * 70)
print(f"  Net flow MAE delta (XGB − LGBM)        : {mae_delta:.4f} bikes/station-hour")
print(f"  Avg revenue rate                       : ${avg_rev_rate:.4f}/trip")
print(f"  Revenue at risk improvement            : ${rev_risk_improvement:>10,.2f}")
print(f"  Rebalancing dispatch saving            : ${xgb_rebal_cost - lgbm_rebal_cost:>10,.2f}")
print(f"  Total estimated LGBM advantage         : ${rev_risk_improvement + (xgb_rebal_cost - lgbm_rebal_cost):>10,.2f}")
print("=" * 70)

spark.createDataFrame(daily).write.mode("overwrite").parquet(FINANCIAL_OUT)
print(f"\nDaily P&L saved to: {FINANCIAL_OUT}")

## Cell 7 — Visualisations

Produces four charts summarising the financial analysis across the full analysis window.

### Plot 1 — Daily Actual Revenue vs Net Flow Prediction MAE
Dual-axis chart showing:
- **Left axis (green):** daily actual revenue in CAD
- **Right axis (blue/orange):** XGB and LGBM daily net flow MAE (bikes/station-hour)

Illustrates how model accuracy tracks alongside revenue seasonality — revealing whether prediction quality degrades during high-demand periods.

### Plot 2 — Cumulative Revenue at Risk: XGB vs LGBM
Cumulative sum of revenue at risk for each model over the analysis window. The shaded area between the two curves represents the cumulative risk advantage of the better model. A wider gap indicates consistently better predictions from LGBM.

### Plot 3 — Daily Rebalancing Dispatch Cost: XGB vs LGBM
Side-by-side bar chart showing the daily estimated cost of incorrect dispatch decisions per model. Each bar pair represents one sampled day (subsampled at ~60 points for readability). Lower bars indicate fewer false-alarm dispatches.

### Plot 4 — Net Flow Prediction Error Distribution
Overlapping histogram of absolute prediction errors (`|predicted − actual|`) for both models across all station-hours. A tighter distribution centred near zero indicates a more reliable model. Mean errors are shown in the legend for direct comparison.


In [0]:

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#F8F9FA",
    "axes.grid":        True,
    "grid.alpha":       0.4,
    "font.family":      "sans-serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
})

BLUE   = "#2E75B6"
ORANGE = "#ED7D31"
GREEN  = "#375623"
RED    = "#C00000"
GREY   = "#7F7F7F"

dates = pd.to_datetime(daily["date"])

fig, axes = plt.subplots(4, 1, figsize=(16, 22))
fig.suptitle(
    f"Bike Share Toronto — Financial Analysis\n"
    f"V3 Direct Net Flow Model (XGBoost vs LightGBM) | {ANALYSIS_START} → {ANALYSIS_END}",
    fontsize=15, fontweight="bold", y=0.99
)

ax = axes[0]
ax2 = ax.twinx()
ax.plot(dates, daily["actual_revenue"], color=GREEN,  lw=2.0, label="Actual Revenue", zorder=3)
ax2.plot(dates, daily["xgb_net_flow_mae"],  color=BLUE,   lw=1.5, label="XGB Net Flow MAE",  linestyle="--", alpha=0.85)
ax2.plot(dates, daily["lgbm_net_flow_mae"], color=ORANGE, lw=1.5, label="LGBM Net Flow MAE", linestyle="-.", alpha=0.85)
ax.set_title("Daily Actual Revenue vs Net Flow Prediction MAE (XGB vs LGBM)")
ax.set_ylabel("Revenue ($ CAD)", color=GREEN)
ax2.set_ylabel("Net Flow MAE (bikes/station-hour)", color=BLUE)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, ncol=3)

ax = axes[1]
cum_xgb_risk  = daily["xgb_revenue_at_risk"].cumsum()
cum_lgbm_risk = daily["lgbm_revenue_at_risk"].cumsum()
ax.plot(dates, cum_xgb_risk,  color=BLUE,   lw=2.0, label="XGB cumulative revenue at risk",  linestyle="--")
ax.plot(dates, cum_lgbm_risk, color=ORANGE, lw=2.0, label="LGBM cumulative revenue at risk", linestyle="-.")
ax.fill_between(dates, cum_lgbm_risk, cum_xgb_risk,
    alpha=0.12, color=GREEN, label="LGBM lower risk")
ax.set_title("Cumulative Revenue At Risk from Net Flow Prediction Error")
ax.set_ylabel("Cumulative Revenue At Risk ($ CAD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.legend(fontsize=9)

ax = axes[2]
bar_width = 0.4
x_idx = range(len(daily))
step  = max(1, len(daily) // 60)
idx   = list(range(0, len(daily), step))
ax.bar([x_idx[i] - bar_width/2 for i in idx],
       [daily_rebal["xgb_rebal_cost"].iloc[i] for i in idx],
       width=bar_width, color=BLUE,   alpha=0.8, label="XGB rebalancing cost")
ax.bar([x_idx[i] + bar_width/2 for i in idx],
       [daily_rebal["lgbm_rebal_cost"].iloc[i] for i in idx],
       width=bar_width, color=ORANGE, alpha=0.8, label="LGBM rebalancing cost")
ax.set_title("Daily Rebalancing Dispatch Cost: XGBoost vs LightGBM (V3 Net Flow)")
ax.set_ylabel("Rebalancing Cost ($ CAD)")
ax.set_xticks([x_idx[i] for i in idx])
ax.set_xticklabels(
    [str(daily["date"].iloc[i])[:10] for i in idx],
    rotation=45, ha="right", fontsize=8
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(fontsize=9)

ax = axes[3]
ax.hist(fin["xgb_net_flow_error"],  bins=50, color=BLUE,   alpha=0.6,
        label=f"XGB  (mean error: {fin['xgb_net_flow_error'].mean():.2f} bikes)", edgecolor="white")
ax.hist(fin["lgbm_net_flow_error"], bins=50, color=ORANGE, alpha=0.6,
        label=f"LGBM (mean error: {fin['lgbm_net_flow_error'].mean():.2f} bikes)", edgecolor="white")
ax.axvline(0, color=GREY, lw=1.5, linestyle="--", label="Perfect prediction")
ax.set_title("Distribution of Net Flow Prediction Error (|predicted − actual|) — V3 Model")
ax.set_xlabel("Absolute Net Flow Error (bikes)")
ax.set_ylabel("Station-Hours")
ax.legend(fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig("/tmp/financial_analysis_v3_netflow.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plots rendered.")

## Cell 8 — Gamification Savings Simulation

Estimates the annual operational cost savings from a **points-based app** that incentivises annual members to return bikes to low-stock (critical) stations, reducing the number of truck dispatches required.

### Points Structure
| Trip type | Points awarded |
|---|---|
| 🟢 Well-stocked → 🔴 Critical | 15 pts |
| 🟢 Well-stocked → 🟡 Low stock | 10 pts |
| 🟡 Low stock → 🔴 Critical | 8 pts |

### Constants
All fixed inputs are sourced from the Bike Share Toronto 2024 Business Review and the Cell 5 rebalancing derivation:
- **Annual rides:** 6,900,000 | **Member share:** 76%
- **Dispatch cost/event (CAD):** derived as `REBAL_ANNUAL_CAD ÷ total annual dispatches` (~\$263.76)
- **Annual dispatches:** ~41,600 (annualised from Cell 5 actual BST event count)

### Simulation Assumptions
| Parameter | Base Case | Description |
|---|---|---|
| App adoption rate | 40% | Share of annual members using the app |
| Redirect rate | 15% | Share of app-user trips redirected to critical stations |
| Trip mix | 20/35/45% | Distribution across green→red, green→yellow, yellow→red |
| Point value | \$0.05 CAD/pt | Monetary value of each reward point |
| Bikes per dispatch | 30 | Average bikes moved per truck run |
| Engagement uplift | 3% | Extra member trips driven by gamification engagement |
| **Rebalancing effectiveness** | **10%** | Fraction of redirected trips that actually prevent a dispatch |

### Rebalancing Effectiveness Factor
This factor accounts for real-world friction that limits user-driven rebalancing:
- **Temporal misalignment:** truck may already be dispatched before a user arrives
- **Geographic mismatch:** no app user may be near the critical station
- **Partial fulfillment:** station needs 8 bikes but only 1–2 users redirect
- **Overlap:** multiple users redirect to the same station, saving only one dispatch

> **Benchmark:** NYC Citi Bike Angels program reports ~5–15% reduction in truck dispatches from user incentive programs.

| Scenario | Effectiveness |
|---|---|
| Pessimistic | 5% |
| Base case | 10% |
| Optimistic | 20% |

### Simulation Outputs
- **Base case P&L:** gross truck savings, points program cost, engagement uplift benefit, and net annual saving
- **Three-scenario comparison:** pessimistic / base / optimistic across dispatch avoided, net saving, and ROI
- **Sensitivity analysis:** net saving heatmap across app adoption rate (10–100%) × redirect rate (5–50%) at base effectiveness

### Visualisations
1. **Scenario bar chart** — gross savings, points cost, and net saving across three scenarios
2. **Base case waterfall** — breakdown of truck savings, points cost, uplift, and net saving
3. **Sensitivity heatmap** — net saving (\$k CAD) across adoption × redirect rate combinations


In [0]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

ANNUAL_RIDES          = 6_900_000
ANNUAL_MEMBER_SHARE   = 0.76
REVENUE_PER_TRIP_ANN  = 0.99
COST_PER_TRIP_VAR     = 1.98
DISPATCH_COST_CAD     = 263.76
N_STATIONS            = 1_008

REBAL_ANNUAL_CAD  = 667 * 1.36 * N_STATIONS * 12   # ~$10,972,523 CAD
ANNUAL_DISPATCHES = round(REBAL_ANNUAL_CAD / DISPATCH_COST_CAD)   # ~41,600

ANNUAL_MEMBER_RIDES = ANNUAL_RIDES * ANNUAL_MEMBER_SHARE   # 5,244,000

print(f"  Annual dispatch budget (CAD) : ${REBAL_ANNUAL_CAD:>12,.2f}")
print(f"  Dispatch cost/event (CAD)   : ${DISPATCH_COST_CAD:>12,.2f}")
print(f"  Total annual dispatches     : {ANNUAL_DISPATCHES:>12,}")
print(f"  Dispatches/station/year     : {ANNUAL_DISPATCHES/N_STATIONS:>12.1f}")

POINTS = {
    "green_to_red":    15,
    "green_to_yellow": 10,
    "yellow_to_red":    8,
}


APP_ADOPTION_RATE  = 0.40    # % of annual members using the app
REDIRECT_RATE      = 0.15    # % of app user trips redirected to critic station

TRIP_MIX = {
    "green_to_red":    0.20,
    "green_to_yellow": 0.35,
    "yellow_to_red":   0.45,
}

POINT_VALUE_CAD    = 0.05    # CAD value of 1 point
BIKES_PER_DISPATCH = 30       # avg bikes carried per truck run (industry: 6–10)
ENGAGEMENT_UPLIFT  = 0.03    # extra trips from gamification engagement (% of member rides)

# What fraction of redirected trips actually prevent a truck dispatch?
# Accounts for:
#   - Temporal misalignment: truck already dispatched before user arrives
#   - Geographic mismatch: no app user near the critic station
#   - Partial fulfillment: station needs 8 bikes, only 1–2 users redirect
#   - Overlap: multiple users redirect to same station, only 1 dispatch saved
#
# Real-world benchmark: NYC Citi Bike Angels program
#   → ~5–15% reduction in truck dispatches from user incentive programs
#
# Scenarios:
#   Pessimistic : 0.05  (~5%  effectiveness)
#   Base case   : 0.10  (~10% effectiveness)  ← DEFAULT
#   Optimistic  : 0.20  (~20% effectiveness)
REBALANCING_EFFECTIVENESS = 0.10


def run_simulation(
    app_adoption      = APP_ADOPTION_RATE,
    redirect_rate     = REDIRECT_RATE,
    trip_mix          = TRIP_MIX,
    point_value       = POINT_VALUE_CAD,
    bikes_per_dispatch= BIKES_PER_DISPATCH,
    effectiveness     = REBALANCING_EFFECTIVENESS,
    uplift            = ENGAGEMENT_UPLIFT,
):
    assert abs(sum(trip_mix.values()) - 1.0) < 1e-6, "TRIP_MIX must sum to 1.0"

    # 1. Redirected trips
    app_rides        = ANNUAL_MEMBER_RIDES * app_adoption
    total_redirected = app_rides * redirect_rate
    redirected_by_type = {tt: total_redirected * mix for tt, mix in trip_mix.items()}

    # 2. Points program cost
    points_cost_by_type = {
        tt: redirected_by_type[tt] * POINTS[tt] * point_value
        for tt in POINTS
    }
    total_points_cost = sum(points_cost_by_type.values())

    # 3. Truck dispatch savings — with effectiveness cap
    #    Only (effectiveness) fraction of redirected trips
    #    actually prevent a dispatch. Then divide by bikes/dispatch
    #    to convert trips → avoided dispatch events.
    effective_trips        = total_redirected * effectiveness
    raw_dispatches_avoided = effective_trips / bikes_per_dispatch
    dispatches_avoided     = min(raw_dispatches_avoided, ANNUAL_DISPATCHES)
    pct_avoided            = dispatches_avoided / ANNUAL_DISPATCHES * 100
    gross_truck_savings    = dispatches_avoided * DISPATCH_COST_CAD

    # 4. Engagement uplift
    uplift_trips   = ANNUAL_MEMBER_RIDES * uplift
    uplift_net     = uplift_trips * (REVENUE_PER_TRIP_ANN - COST_PER_TRIP_VAR)
    uplift_benefit = max(0.0, uplift_net)

    # 5. Net saving
    net_saving = gross_truck_savings - total_points_cost + uplift_benefit
    roi_pct    = (gross_truck_savings / total_points_cost - 1) * 100 \
                 if total_points_cost > 0 else float("inf")

    return {
        "app_rides":              app_rides,
        "total_redirected":       total_redirected,
        "redirected_by_type":     redirected_by_type,
        "points_cost_by_type":    points_cost_by_type,
        "total_points_cost":      total_points_cost,
        "effective_trips":        effective_trips,
        "raw_dispatches_avoided": raw_dispatches_avoided,
        "dispatches_avoided":     dispatches_avoided,
        "pct_avoided":            pct_avoided,
        "gross_truck_savings":    gross_truck_savings,
        "uplift_benefit":         uplift_benefit,
        "net_saving":             net_saving,
        "roi_pct":                roi_pct,
    }


r = run_simulation()

print()
print("=" * 65)
print(" CELL 8 — GAMIFICATION SAVINGS SIMULATION")
print("=" * 65)
print(f" Annual member rides          : {ANNUAL_MEMBER_RIDES:>12,.0f}")
print(f" App adoption rate            : {APP_ADOPTION_RATE*100:>11.1f}%")
print(f" App user rides/yr            : {r['app_rides']:>12,.0f}")
print(f" Redirect rate                : {REDIRECT_RATE*100:>11.1f}%")
print(f" Total redirected trips/yr    : {r['total_redirected']:>12,.0f}")
print(f" Rebalancing effectiveness    : {REBALANCING_EFFECTIVENESS*100:>11.1f}%")
print(f" Effective trips (→ dispatch) : {r['effective_trips']:>12,.0f}")
print()
print(f" {'Trip type':<26} {'Redirected trips':>16}  {'Points cost ($)':>15}")
print("-" * 65)
for tt, trips in r["redirected_by_type"].items():
    label = (tt.replace("green_to_red",    "🟢 → 🔴 (15 pts)")
               .replace("green_to_yellow", "🟢 → 🟡 (10 pts)")
               .replace("yellow_to_red",   "🟡 → 🔴  (8 pts)"))
    print(f" {label:<26} {trips:>16,.0f}  ${r['points_cost_by_type'][tt]:>14,.2f}")
print("-" * 65)
print(f" {'TOTAL':<26} {r['total_redirected']:>16,.0f}  ${r['total_points_cost']:>14,.2f}")
print()
print(f" {'Metric':<48} {'Value':>14}")
print("-" * 65)
print(f" {'Total annual dispatches (system)':<48} {ANNUAL_DISPATCHES:>14,}")
print(f" {'Bikes per dispatch (assumed)':<48} {BIKES_PER_DISPATCH:>14}")
print(f" {'Rebalancing effectiveness factor':<48} {REBALANCING_EFFECTIVENESS*100:>13.1f}%")
print(f" {'Effective trips (after effectiveness discount)':<48} {r['effective_trips']:>14,.0f}")
print(f" {'Raw dispatches avoided (eff. trips ÷ bikes)':<48} {r['raw_dispatches_avoided']:>14,.0f}")
print(f" {'Dispatches avoided (capped at annual total)':<48} {r['dispatches_avoided']:>14,.0f}")
print(f" {'% of annual dispatches eliminated':<48} {r['pct_avoided']:>13.1f}%")
print(f" {'Gross truck savings (CAD)':<48} ${r['gross_truck_savings']:>13,.2f}")
print(f" {'Total points program cost (CAD)':<48} ${r['total_points_cost']:>13,.2f}")
print(f" {'Engagement uplift benefit (CAD)':<48} ${r['uplift_benefit']:>13,.2f}")
print("-" * 65)
print(f" {'NET ANNUAL SAVING (CAD)':<48} ${r['net_saving']:>13,.2f}")
print(f" {'ROI on points investment':<48} {r['roi_pct']:>13.1f}%")
print("=" * 65)
print()
print(" ASSUMPTIONS USED")
print("-" * 65)
print(f"  App adoption rate           : {APP_ADOPTION_RATE*100:.0f}% of annual members")
print(f"  Redirect rate               : {REDIRECT_RATE*100:.0f}% of app user trips redirected")
print(f"  Trip mix (🟢→🔴)            : {TRIP_MIX['green_to_red']*100:.0f}%  |  "
      f"(🟢→🟡): {TRIP_MIX['green_to_yellow']*100:.0f}%  |  "
      f"(🟡→🔴): {TRIP_MIX['yellow_to_red']*100:.0f}%")
print(f"  Point value                 : ${POINT_VALUE_CAD:.3f} CAD/pt")
print(f"  Bikes per dispatch          : {BIKES_PER_DISPATCH} bikes (industry avg: 6–10)")
print(f"  Rebalancing effectiveness   : {REBALANCING_EFFECTIVENESS*100:.0f}%  "
      f"(Citi Bike Angels benchmark: 5–15%)")
print(f"  Engagement uplift           : {ENGAGEMENT_UPLIFT*100:.1f}% extra member trips")
print(f"  Dispatch cost               : ${DISPATCH_COST_CAD} CAD  (Cell 5 — REBAL_COST_PER_EVENT)")
print(f"  Annual dispatches           : {ANNUAL_DISPATCHES:,}  (Cell 5 budget ÷ dispatch cost)")
print("=" * 65)


scenarios = {
    "Pessimistic": {"effectiveness": 0.05},
    "Base case":   {"effectiveness": 0.10},
    "Optimistic":  {"effectiveness": 0.20},
}

print()
print("=" * 65)
print(" THREE-SCENARIO COMPARISON")
print(f" {'Scenario':<15} {'Effectiveness':>13} {'Disp. avoided':>14} {'% elim.':>8} {'Net saving':>14} {'ROI':>8}")
print("-" * 65)
scenario_results = {}
for name, params in scenarios.items():
    s = run_simulation(effectiveness=params["effectiveness"])
    scenario_results[name] = s
    print(f" {name:<15} {params['effectiveness']*100:>12.0f}% "
          f"{s['dispatches_avoided']:>14,.0f} "
          f"{s['pct_avoided']:>7.1f}% "
          f"${s['net_saving']:>13,.2f} "
          f"{s['roi_pct']:>7.1f}%")
print("=" * 65)


adoption_range = np.arange(0.10, 1.05, 0.10)
redirect_range = np.arange(0.05, 0.55, 0.05)

sensitivity = []
for adopt in adoption_range:
    for redir in redirect_range:
        s = run_simulation(app_adoption=adopt, redirect_rate=redir)
        sensitivity.append({
            "adoption_pct":   round(adopt * 100),
            "redirect_pct":   round(redir * 100),
            "net_saving_cad": round(s["net_saving"], 2),
        })

sens_df = pd.DataFrame(sensitivity)
pivot   = sens_df.pivot(
    index="adoption_pct",
    columns="redirect_pct",
    values="net_saving_cad"
)

print()
print(f" SENSITIVITY — Net Annual Saving ($CAD) | Effectiveness = {REBALANCING_EFFECTIVENESS*100:.0f}%")
print(" Rows = App adoption %, Columns = Redirect rate %")
print(pivot.to_string())


plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#F8F9FA",
    "axes.grid":        True,
    "grid.alpha":       0.4,
    "font.family":      "sans-serif",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
})

GREEN = "#1D9E75"
AMBER = "#BA7517"
RED   = "#D85A30"
BLUE  = "#378ADD"
DARK  = "#0F6E56"
GREY  = "#888780"

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    "Cell 8 — Gamification Savings Simulation\n"
    "Bike Share Toronto | Points-Based Rebalancing App",
    fontsize=14, fontweight="bold"
)

ax = axes[0]
names     = list(scenarios.keys())
net_vals  = [scenario_results[n]["net_saving"]       for n in names]
pt_costs  = [scenario_results[n]["total_points_cost"] for n in names]
truck_sav = [scenario_results[n]["gross_truck_savings"] for n in names]
x = np.arange(len(names))
w = 0.28
ax.bar(x - w, truck_sav, width=w, color=GREEN, alpha=0.85, label="Gross truck savings")
ax.bar(x,    [-c for c in pt_costs], width=w, color=RED,   alpha=0.85, label="Points cost")
ax.bar(x + w, net_vals,  width=w, color=DARK,  alpha=0.85, label="Net saving")
ax.axhline(0, color=GREY, linewidth=0.8, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.set_title("Pessimistic / Base / Optimistic")
ax.set_ylabel("$ CAD")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.legend(fontsize=8)

ax = axes[1]
categories = ["Truck\nsavings", "Points\ncost", "Uplift\nbenefit", "Net\nsaving"]
values     = [
    r["gross_truck_savings"],
    -r["total_points_cost"],
    r["uplift_benefit"],
    r["net_saving"],
]
colors = [GREEN, RED, BLUE, DARK]
bars   = ax.bar(categories, values, color=colors, width=0.55, edgecolor="white", linewidth=0.5)
ax.axhline(0, color=GREY, linewidth=0.8, linestyle="--")
ymax = max(abs(v) for v in values)
for bar, val in zip(bars, values):
    sign = "+" if val >= 0 else ""
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        val + (ymax * 0.03 if val >= 0 else -ymax * 0.03),
        f"{sign}${val:,.0f}",
        ha="center", va="bottom" if val >= 0 else "top",
        fontsize=9, fontweight="500"
    )
ax.set_title(f"Base Case Breakdown\n(effectiveness = {REBALANCING_EFFECTIVENESS*100:.0f}%)")
ax.set_ylabel("$ CAD")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"${v:,.0f}"))

ax = axes[2]
im = ax.imshow(pivot.values / 1000, aspect="auto", cmap="RdYlGn", origin="lower")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{c}%" for c in pivot.columns], fontsize=8, rotation=45)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"{r}%" for r in pivot.index], fontsize=8)
ax.set_xlabel("Redirect rate (% of app user trips)")
ax.set_ylabel("App adoption rate")
ax.set_title(f"Net Saving Sensitivity ($k CAD)\nEffectiveness = {REBALANCING_EFFECTIVENESS*100:.0f}%")
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j] / 1000
        ax.text(j, i, f"{val:.0f}k", ha="center", va="center", fontsize=7, color="black")
plt.colorbar(im, ax=ax, label="Net saving ($k CAD)")

plt.tight_layout()
plt.show()